# Notebook 16 — Model Calibration Analysis

**Owner:** Satya Yadav  
**Phase 2, Step 4**  
**Branch:** `feat/satya-calibration`

---

## Purpose

Calibration asks: *when a model predicts a CCME index of 85, do samples with that
prediction really have a true index of 85 on average?*  A model that answers yes
reliably across the full prediction range is **calibrated**.

Calibration matters operationally because CCME categories carry regulatory meaning:
- **Excellent** (95–100): no management action required
- **Good** (80–94): minor concern
- **Fair** (65–79): occasional impairment
- **Marginal** (45–64): frequent impairment
- **Poor** (0–44): almost always impaired

A model that is well-calibrated overall (low MAE) may still be systematically biased
in the operationally critical Poor/Marginal range.  Calibration analysis surfaces this.

## What this notebook does

1. Loads the shared test predictions from all Phase 1 and Phase 2 models.
2. For each model, runs the full calibration suite from `src/metrics.py`:
   - **Reliability diagram** — mean predicted vs mean true across 10 equal-width bins.
   - **ECE / MCE** — Expected and Maximum Calibration Error (normalised to [0, 1]).
   - **Category calibration** — per-category hit rate and MAE for the five CCME classes.
3. Plots reliability diagrams for all models on a shared axis.
4. Produces a summary table comparing ECE, MCE, and category accuracy across models.

## Implementation notes

All calibration functions live in `src/metrics.py` and are tested in
`tests/test_calibration.py`.  This notebook calls them through the same interface
every other module uses — no metric logic is re-defined here.

> **Relationship to `calibration_data()` (Phase 1):** `calibration_data()` bins
> predictions into CCME categories and returns classification metrics (macro F1,
> per-class F1). The Phase 2 calibration suite goes further: it quantifies *how far*
> from the diagonal the model sits across the continuous prediction range (ECE/MCE)
> and reports per-category regression error (MAE within each class).

---
## 0. Imports and path setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── repo root on sys.path ──────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import FIGURES, REPORTS, SEED
from src.evaluate_api import evaluate, get_splits
from src.metrics import (
    calibration_summary,
    category_calibration_summary,
    expected_calibration_error,
    reliability_diagram_data,
)

FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
RESULTS = REPO_ROOT / 'results'

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150})

print(f'SEED = {SEED}')
print(f'FIGURES -> {FIGURES}')

---
## 1. Load the shared test split

Every model is evaluated on the **same** test partition produced by `src/split.py`.
Only `y_test` is needed for calibration; we do not re-train anything here.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = get_splits()

print('Test set size:', len(y_test))
print('Target range :', y_test.min(), '—', y_test.max())
print('Ceiling share:', f'{(y_test == 100.0).mean():.2%}')

---
## 2. Load saved test-set predictions for each model

Models are scored on the test set exactly once and their raw predictions are saved
alongside the aggregated CSV results.  This notebook reads those saved predictions
so the test set is not evaluated again (no double-dipping).

> If a model's prediction file does not exist, we fall back to reproducing the
> constant naive baseline predictions on the fly (the only model that needs no
> file, because it is deterministic from `y_train`).

In [ ]:
# ── Naive baseline (constant predictor — no file needed) ───────────────────
TRAIN_MEDIAN = float(y_train.median())
TRAIN_MEAN   = float(y_train.mean())

naive_median_pred = np.full(len(y_test), TRAIN_MEDIAN)
naive_mean_pred   = np.full(len(y_test), TRAIN_MEAN)

print(f'Naive median predictor: {TRAIN_MEDIAN:.4f}')
print(f'Naive mean predictor  : {TRAIN_MEAN:.4f}')

# ── Model predictions dictionary ──────────────────────────────────────────
# Keys are display names; values are (y_pred array, is_available)
model_preds: dict[str, np.ndarray | None] = {
    'Naive Median': naive_median_pred,
    'Naive Mean':   naive_mean_pred,
}

# Phase 1 models — load saved predictions if available
for name, fname in [
    ('Random Forest', 'rf_test_predictions.npy'),
    ('XGBoost',       'xgb_test_predictions.npy'),
]:
    p = RESULTS / fname
    if p.exists():
        model_preds[name] = np.load(p)
        print(f'Loaded {name} predictions from {p.name}')
    else:
        print(f'[WARN] {p.name} not found — {name} will be skipped.')
        model_preds[name] = None

# Phase 2 models
for name, fname in [
    ('Base MLP',      'mlp_test_predictions.npy'),
    ('Hybrid',        'hybrid_test_predictions.npy'),
]:
    p = RESULTS / fname
    if p.exists():
        model_preds[name] = np.load(p)
        print(f'Loaded {name} predictions from {p.name}')
    else:
        print(f'[WARN] {p.name} not found — {name} will be skipped.')
        model_preds[name] = None

# Keep only models with predictions
available_models = {k: v for k, v in model_preds.items() if v is not None}
print(f'\nModels available for calibration: {list(available_models.keys())}')

---
## 3. Calibration suite — run for all models

`calibration_summary()` is a single call that returns:
- **`regression_metrics`** — MAE, RMSE, R², MAPE, MedAE.
- **`ece_metrics`** — ECE, MCE, mean gap (all normalised to [0, 1]).
- **`reliability_diagram`** — per-bin mean predicted and mean true values.
- **`category_calibration`** — per-CCME-class hit rate, MAE, mean predicted.

In [ ]:
N_BINS = 10
y_test_arr = y_test.values

summaries: dict[str, dict] = {}
for name, y_pred in available_models.items():
    summaries[name] = calibration_summary(y_test_arr, y_pred, n_bins=N_BINS)
    ece = summaries[name]['ece_metrics']['ece']
    mce = summaries[name]['ece_metrics']['mce']
    acc = summaries[name]['category_calibration']['overall_category_accuracy']
    mae = summaries[name]['regression_metrics']['mae']
    print(f'{name:<20}  MAE={mae:6.4f}  ECE={ece:.4f}  MCE={mce:.4f}  CatAcc={acc:.4f}')

---
## 4. Reliability diagrams

Each panel shows mean predicted value (x-axis) vs mean true value (y-axis) across
10 equal-width bins.  The dashed diagonal is the perfect-calibration reference.
Points above the diagonal indicate **under-prediction** (the model predicts lower
than the true value); points below indicate **over-prediction**.

In [ ]:
n_models = len(available_models)
ncols = min(3, n_models)
nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
ax_list = [axes[r][c] for r in range(nrows) for c in range(ncols)]

CCME_CATEGORY_BOUNDS = [0, 44, 64, 79, 94, 100]

for ax, (name, y_pred) in zip(ax_list, available_models.items()):
    rd = summaries[name]['reliability_diagram']
    ece = summaries[name]['ece_metrics']['ece']

    xs = [mp for mp in rd['mean_predicted'] if mp is not None]
    ys = [mt for mt in rd['mean_true']      if mt is not None]
    ns = [n  for mp, n in zip(rd['mean_predicted'], rd['bin_counts']) if mp is not None]

    if xs:
        sc = ax.scatter(xs, ys, s=[max(20, n / 500) for n in ns],
                        c=ns, cmap='viridis', alpha=0.85, zorder=3)
        plt.colorbar(sc, ax=ax, label='Bin count')

    # Perfect-calibration diagonal
    ax.plot([0, 100], [0, 100], 'k--', lw=1.2, label='Perfect calibration', zorder=2)

    # CCME category boundaries
    for bound in CCME_CATEGORY_BOUNDS[1:-1]:
        ax.axvline(bound, color='grey', lw=0.6, ls=':', alpha=0.6)
        ax.axhline(bound, color='grey', lw=0.6, ls=':', alpha=0.6)

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Mean Predicted CCME Value')
    ax.set_ylabel('Mean True CCME Value')
    ax.set_title(f'{name}\nECE = {ece:.4f}' if not (ece != ece) else f'{name}\nECE = N/A')
    ax.legend(fontsize=8)

# Hide unused axes
for ax in ax_list[n_models:]:
    ax.set_visible(False)

plt.suptitle('Reliability Diagrams — All Models (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES / '16_reliability_diagrams.png', bbox_inches='tight')
plt.show()
print(f'Saved -> {FIGURES / "16_reliability_diagrams.png"}')

---
## 5. ECE / MCE summary table

ECE and MCE are both normalised by the index range (0–100) so they lie in [0, 1].
Lower is better.  ECE is sample-weighted across bins; MCE is the worst bin.

In [ ]:
rows = []
for name, s in summaries.items():
    rm = s['regression_metrics']
    em = s['ece_metrics']
    cc = s['category_calibration']
    rows.append({
        'Model':           name,
        'MAE':             round(rm['mae'],   4),
        'RMSE':            round(rm['rmse'],  4),
        'R²':              round(rm['r2'],    6),
        'ECE (norm)':      round(em['ece'],   6) if not (em['ece'] != em['ece']) else float('nan'),
        'MCE (norm)':      round(em['mce'],   6) if not (em['mce'] != em['mce']) else float('nan'),
        'Cat. Accuracy':   round(cc['overall_category_accuracy'], 4),
    })

df_summary = pd.DataFrame(rows).set_index('Model')
print('=== Calibration Summary Table (Test Set) ===')
print(df_summary.to_string())

# Save to CSV
out_path = RESULTS / 'calibration_summary.csv'
df_summary.to_csv(out_path)
print(f'\nSaved -> {out_path}')

---
## 6. Per-category calibration analysis

For each CCME WQI class we report:
- **True N** — how many test records truly belong to this class.
- **Hit Rate** — fraction of those records correctly classified (per-class recall).
- **MAE** — mean absolute regression error restricted to this class.

The **Poor** class (index 0–44) is the operationally most critical: a model that
fails here may misclassify heavily impaired water as compliant.

In [ ]:
CCME_CATS = ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']

for name, s in summaries.items():
    cc = s['category_calibration']['categories']
    print(f'\n── {name} ──')
    print(f'{"Category":<12} {"True N":>8} {"Hit Rate":>10} {"MAE":>8} {"Mean Pred":>10} {"Mean True":>10}')
    print('-' * 62)
    for cat in CCME_CATS:
        d = cc[cat]
        hr = f'{d["hit_rate"]:.4f}' if not (d['hit_rate'] != d['hit_rate']) else '   N/A'
        mae = f'{d["mae"]:.4f}'    if not (d['mae'] != d['mae'])       else '   N/A'
        mp = f'{d["mean_pred"]:.2f}' if not (d['mean_pred'] != d['mean_pred']) else '  N/A'
        mt = f'{d["mean_true"]:.2f}' if not (d['mean_true'] != d['mean_true']) else '  N/A'
        print(f'{cat:<12} {d["true_n"]:>8,} {hr:>10} {mae:>8} {mp:>10} {mt:>10}')

---
## 7. Per-category hit-rate bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(CCME_CATS))
width = 0.8 / len(available_models)
colors = sns.color_palette('muted', len(available_models))

for i, (name, s) in enumerate(summaries.items()):
    cc = s['category_calibration']['categories']
    hit_rates = [
        cc[cat]['hit_rate'] if not (cc[cat]['hit_rate'] != cc[cat]['hit_rate']) else 0.0
        for cat in CCME_CATS
    ]
    offset = (i - len(available_models) / 2 + 0.5) * width
    bars = ax.bar(x + offset, hit_rates, width * 0.9,
                  label=name, color=colors[i], alpha=0.85)

ax.axhline(1.0, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(CCME_CATS)
ax.set_xlabel('CCME WQI Category')
ax.set_ylabel('Category Hit Rate (Recall)')
ax.set_title('Per-Category Hit Rate — All Models (Test Set)')
ax.set_ylim(0, 1.05)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
fig.savefig(FIGURES / '16_category_hit_rates.png', bbox_inches='tight')
plt.show()
print(f'Saved -> {FIGURES / "16_category_hit_rates.png"}')

---
## 8. Calibration gap plot

The calibration gap is `mean_predicted - mean_true` for each bin.  A positive gap
means the model over-predicts in that region; negative means under-prediction.  Gaps
near zero across all bins indicate good calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('muted', len(available_models))

for (name, s), color in zip(summaries.items(), colors):
    rd = s['reliability_diagram']
    centers = [
        c for c, mp in zip(rd['bin_centers'], rd['mean_predicted'])
        if mp is not None
    ]
    gaps = [
        g for g in rd['calibration_gaps']
        if g is not None
    ]
    if centers:
        ax.plot(centers, gaps, 'o-', color=color, label=name, alpha=0.85)

ax.axhline(0, color='black', lw=1.0, ls='--', alpha=0.5, label='Zero gap')

# CCME category boundaries
for bound, label in zip([44, 64, 79, 94], ['Poor|Marg', 'Marg|Fair', 'Fair|Good', 'Good|Exc']):
    ax.axvline(bound, color='grey', lw=0.7, ls=':', alpha=0.5)
    ax.text(bound + 0.5, ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] else 1,
            label, fontsize=7, color='grey', rotation=90, va='top')

ax.set_xlabel('Bin Centre (CCME Index)')
ax.set_ylabel('Calibration Gap (Predicted − True, index pts)')
ax.set_title('Calibration Gaps Across Prediction Range — All Models')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
fig.savefig(FIGURES / '16_calibration_gaps.png', bbox_inches='tight')
plt.show()
print(f'Saved -> {FIGURES / "16_calibration_gaps.png"}')

---
## 9. Notebook checklist

- [x] Calibration functions imported from `src/metrics.py` — no logic re-defined here.
- [x] All models evaluated on the **same** shared test partition (zero re-evaluation).
- [x] Reliability diagram saved to `reports/figures/16_reliability_diagrams.png`.
- [x] Category hit-rate chart saved to `reports/figures/16_category_hit_rates.png`.
- [x] Calibration gap chart saved to `reports/figures/16_calibration_gaps.png`.
- [x] Summary table saved to `results/calibration_summary.csv`.
- [x] ECE and MCE are normalised — directly comparable across models.
- [x] Per-category analysis highlights the operationally critical Poor class.